# Explainable search engine

In [ ]:
import os, sys, logging
logging.basicConfig(stream=sys.stdout, 
                    format='%(asctime)s %(message)s',
                    level=logging.INFO)

In [ ]:
lang = "en"

## load DBpedia data

In [ ]:
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore
from rdflib import URIRef
from nifigator import NifGraph

database_url = 'http://localhost:3030/dbpedia_'+lang
identifier = URIRef("https://mangosaurus.eu/dbpedia")

# Connect to triplestore
store = SPARQLUpdateStore(
    query_endpoint = database_url+'/sparql',
    update_endpoint = database_url+'/update'
)
# Create NifVectorGraph with this store
g = NifGraph(
    store=store, 
    identifier=identifier
)

Import the vector representations of the phrases, lemmas and contexts


In [ ]:
from selmr import SELMR, LanguageMultisets, STOPWORDS

# set the parameters to create the SELMR instance
params = {
    "words_filter": {
        "data": STOPWORDS['en'],
        "name": "selmr.STOPWORDS"
    },
    "uncased": True, 
    "lemmatized": False
}

# construct a SELMR data structure with the DBpedia phrases and contexts
selmr = SELMR(
    path="..//data//dbpedia_10000",
    params=params
)

### Load and set up two DBpedia pages

We take two pages from DBpedia about two stars, Aldebaran and Antares. 

In [ ]:
# Read two documents in DBpedia about Aldebaran and Antares stars
doc1 = g.get(
    URIRef("http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=context")
)
doc2 = g.get(
    URIRef("http://dbpedia.org/resource/Antares?dbpv=2020-07&nif=context")
)

The first document reads:

In [ ]:
print(doc1)

```console
(nif:Context) uri = <http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=context>
  sourceUrl : <http://en.wikipedia.org/wiki/Aldebaran?oldid=964792900&ns=0>
  predLang : <http://lexvo.org/id/iso639-3/eng>
  isString : 'Aldebaran , designated α Tauri (Latinized to Alpha Tauri, abbreviated Alpha Tau, α Tau), is an orange giant star measured to be about 65 light-years from the Sun in the zodiac constellation Taurus. It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun\'s, so it is over 400 times as luminous. It spins slowly and takes 520 days to complete a rotation. The planetary exploration probe Pioneer 10 is heading in the general direction of the star and should make its closest approach in about two million years.\n\nNomenclature\nThe traditional name Aldebaran derives from the Arabic al Dabarān, meaning "the follower", because it seems to follow the Pleiades. In 2016, the I... '
  firstSentence : 'Aldebaran , designated α Tauri (Latinized to Alpha Tauri, abbreviated Alpha Tau, α Tau), is an orange giant star measured to be about 65 light-years from the Sun in the zodiac constellation Taurus.'
  lastSentence : '* Daytime occultation of Aldebaran by the Moon (Moscow, Russia) YouTube video'
```

THe second document reads:

In [ ]:
print(doc2)

```console
(nif:Context) uri = <http://dbpedia.org/resource/Antares?dbpv=2020-07&nif=context>
  sourceUrl : <http://en.wikipedia.org/wiki/Antares?oldid=964919229&ns=0>
  predLang : <http://lexvo.org/id/iso639-3/eng>
  isString : 'Antares , designated α Scorpii (Latinised to Alpha Scorpii, abbreviated Alpha Sco, α Sco), is on average the fifteenth-brightest star in the night sky, and the brightest object in the constellation of Scorpius. Distinctly reddish when viewed with the naked eye, Antares is a slow irregular variable star that ranges in brightness from apparent magnitude +0.6 to +1.6. Often referred to as "the heart of the scorpion", Antares is flanked by σ Scorpii and τ Scorpii near the center of the constellation. Classified as spectral type M1.5Iab-Ib, Antares is a red supergiant, a large evolved massive star and one of the largest stars visible to the naked eye. Its exact size remains uncertain, but if placed at the center of the Solar System, it would reach to somewhere between the orbits of Mars and Jupiter. Its mass is calculated to be around 12 times that of the Sun. Antares is the brightest, most massive, and most evolved stellar member of the nearest OB association, the Scorpius–Centaurus Associ... '
  firstSentence : 'Antares , designated α Scorpii (Latinised to Alpha Scorpii, abbreviated Alpha Sco, α Sco), is on average the fifteenth-brightest star in the night sky, and the brightest object in the constellation of Scorpius.'
  lastSentence : '* Best Ever Image of a Star’s Surface and Atmosphere - First map of motion of material on a star other than the Sun'
```

## Find similar sentences

For sentences similarities we sum the contexts of the all the phrases in the sentences, thereby obtaining a multiset representation of the sentence. Then we calculate the Jaccard distance between the sentences and sort with increasing distance.

The Jaccard index is

```{math}
J(A, B) = \frac { | A \bigcap B |} { |A \bigcup B| }
```

Create a vector of every sentences of both documents.


In [ ]:
# setup dictionary with sentences and their vector representation
doc1_vector = {
    sent.anchorOf: selmr.derive_multisets(
        document=sent.anchorOf,
        merge_dict=True,
    )
    for sent in doc1.sentences
}
doc2_vector = {
    sent.anchorOf: selmr.derive_multisets(
        document=sent.anchorOf,
        merge_dict=True,
    )
    for sent in doc2.sentences
}

Calculate the distances (based on Jaccard index) of all sentence combinations of first and second document.


In [ ]:
from selmr.multisets import jaccard_index, merge_multiset, weighted_jaccard_index

# Calculate the Jaccard distance for all sentence combinations
d = {
    (s1, s2): 1-jaccard_index(c1, c2)
    for s1, c1 in doc1_vector.items()
    for s2, c2 in doc2_vector.items()
}
# Sort the results with lowest distance
similarities = sorted(d.items(), key=lambda item: item[1], reverse=False)

Print the results


In [ ]:
# print the results
for item in similarities[0:5]:
    print(repr(item[0][0]) + " \n<- distance = {0:.4f}".format(float(item[1]))+" ->\n"+repr(item[0][1])+"\n")

This is the output:

```console
'External links' 
<- distance = 0.0000 ->
'External links'

'It is a variable star listed in the General Catalogue of Variable Stars, but it is listed using its Bayer designation and does not have a separate variable star designation.' 
<- distance = 0.2794 ->
'Antares is a variable star and is listed in the General Catalogue of Variable Stars but as a Bayer-designated star it does not have a separate variable star designation.'

'Aldebaran is 5.47 degrees south of the ecliptic and so can be occulted by the Moon.' 
<- distance = 0.5952 ->
'Occultations and conjunctions\nAntares is 4.57 degrees south of the ecliptic, one of four first magnitude stars within 6° of the ecliptic (the others are Spica, Regulus and Aldebaran), so it can be occulted by the Moon.'

'Aldebaran is the brightest star in the constellation Taurus and so has the Bayer designation α Tauri, Latinised as Alpha Tauri.' 
<- distance = 0.6106 ->
"Nomenclature\nα Scorpii (Latinised to Alpha Scorpii) is the star's Bayer designation."

'The angular diameter of Aldebaran has been measured many times.' 
<- distance = 0.6384 ->
'An apparent diameter from occultations 41.3 ± 0.1 milliarcseconds has been published.'


``` 

## Explainable text search

For text search we need another distance function. Now we are interested in the extent to which the vector of a sentence contains the vector representation of a query. For this we use the containment or support, defined by the cardinality of the intersection between A and B divided by the cardinality of A.

```{math}
containment(A, B) = \frac { | A \bigcap B |} { |A| }
```

The sentence with the highest containment has the most contexts in common and thus is the closest to the text.

In [ ]:
from selmr import SELMR, LanguageMultisets, STOPWORDS

params = {
    "words_filter": {
        "data": STOPWORDS['en'],
        "name": "selmr.STOPWORDS"
    },
    "uncased": True, 
    "lemmatized": True
}
# construct a SELMR data structure with the DBpedia phrases and contexts
selmr = SELMR(
    path="..//data//dbpedia_10000",
    params=params
)

In [ ]:
# setup dictionary with sentences and their contexts
v_doc_lemmas = {
    (sent.uri, sent.anchorOf): selmr.derive_multisets(
        document=sent.anchorOf
    )
    for sent in doc1.sentences+doc2.sentences
}

### Using the MinHashSearch 

In [ ]:
# from selmr import MinHashSearch

# mhs = MinHashSearch(
#     selmr=selmr,
#     documents=v_doc_lemmas
# )

In [ ]:
# import pickle
# with open('..\\data\\minhash.pickle', 'wb') as fh:
#     pickle.dump(mhs.minhash_dict, fh)

In [ ]:
import pickle
with open('..\\data\\minhash.pickle', 'rb') as fh:
    minhash_dict = pickle.load(fh)

In [ ]:
from nifigator import jaccard_index

s1 = 'large'
s2 = 'small'
print("estimated Jaccard index: "+str(minhash_dict[s2].jaccard(minhash_dict[s1])))
print("actual Jaccard index: "+str(float(jaccard_index(
    set(p[0] for p in selmr.contexts(s1).most_common(15)),
    set(p[0] for p in selmr.contexts(s2).most_common(15)),
))))

This shows:

```console
estimated Jaccard index: 0.3515625
actual Jaccard index: 0.36363636363636365
```

In [ ]:
from selmr import MinHashSearch

mhs = MinHashSearch(
    selmr=selmr,
    minhash_dict=minhash_dict,
    documents=v_doc_lemmas,
)

### Example with two DBpedia pages

In [ ]:
query = "brightest star in the Taurus cluster"
mhs.print_search_results(query, topn=3)

These are the results:

```console
http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=sentence_198_624
"It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun's, so it is over 400 times as luminous."
- Score 0.1233: 
- Full matches:
    'brightest'
    'brightest star'
    'star'
    'Taurus'
- Close matches:
    'cluster' -> 'sky' (0.8000)
    'cluster' -> 'planet' (0.8000)
    'cluster' -> 'mass' (0.8000)
    'cluster' -> 'sun' (0.8000)
    'cluster' -> 'surface' (0.8000)
    'cluster' -> 'temperature' (0.8000)
    'cluster' -> 'radius' (0.8000)
    'cluster' -> 'star' (0.8667)
    'cluster' -> 'night' (0.8667)
    'cluster' -> 'night sky' (0.8667)
    'cluster' -> 'magnitude' (0.8667)
    'cluster' -> 'host' (0.8667)
    'cluster' -> 'times' (0.8667)
    'cluster' -> 'named' (0.8667)
    'cluster' -> 'brightness' (0.9333)
    'cluster' -> 'several' (0.9333)
    'cluster' -> 'red giant' (0.9333)
    'cluster' -> 'giant' (0.9333)
    'cluster' -> 'surface temperature' (0.9333)
http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=sentence_1117_1244
'Aldebaran is the brightest star in the constellation Taurus and so has the Bayer designation α Tauri, Latinised as Alpha Tauri.'
- Score 0.1644: 
- Full matches:
    'brightest'
    'brightest star'
    'star'
    'Taurus'
- Close matches:
    'cluster' -> 'constellation' (0.8000)
    'cluster' -> 'star' (0.8667)
    'cluster' -> 'designation' (0.9333)
http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=sentence_14378_14480
'As the brightest star in a Zodiac constellation, it is also given great significance within astrology.'
- Score 0.3014: 
- Full matches:
    'brightest'
    'brightest star'
    'star'
- Close matches:
    'cluster' -> 'Zodiac' (0.8000)
    'cluster' -> 'constellation' (0.8000)
    'cluster' -> 'star' (0.8667)
    'cluster' -> 'great' (0.8667)
    'cluster' -> 'significance' (0.8667)
    'Taurus' -> 'astrology' (0.8667)
    'Taurus' -> 'it' (0.9333)
```

In [ ]:
query = "Herschel discover Aldebaran"
mhs.print_search_results(query, topn=3)

These are the results:

```console
http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=sentence_4262_4405
'English astronomer William Herschel discovered a faint companion to Aldebaran in 1782; an 11th magnitude star at an angular separation of 117″.'
- Score 0.0000: 
- Full matches:
    'Herschel'
    'discovered'
    'Aldebaran'
http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=sentence_4575_4723
"Follow on measurements of proper motion showed that Herschel's companion was diverging from Aldebaran, and hence they were not physically connected."
- Score 0.2927: 
- Full matches:
    'Herschel'
    'Aldebaran'
- Close matches:
    'discover' -> 'Follow' (0.8667)
    'discover' -> 'showed' (0.8667)
    'discover' -> 'showed that' (0.9333)
    'discover' -> 'was' (0.9333)
    'discover' -> 'were' (0.9333)
    'discover' -> 'connected' (0.9333)
http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=sentence_4724_4882
'However, the companion discovered by Burnham had almost exactly the same proper motion as Aldebaran, suggesting that the two formed a wide binary star system.'
- Score 0.3659: 
- Full matches:
    'discovered'
    'Aldebaran'
```

### Example with 20 DBpedia pages

In [ ]:
# setup dictionary with sentences and their contexts
from rdflib import RDF
from nifigator import NIF

v_doc_lemmas = dict()
for url in list(g.subjects(RDF.type, NIF.Context))[:10]:
    context = g.get(URIRef(url))
    if context.isString is not None:
        context.extract_sentences(forced_sentence_split_characters=["*"])
        for sent in context.sentences:
            v_doc_lemmas[(sent.uri, sent.anchorOf)] = selmr.derive_multisets(
                document=sent.anchorOf
            )

In [ ]:
from selmr import MinHashSearch

mhs = MinHashSearch(
    selmr=selmr,
    minhash_dict=minhash_dict,
    documents=v_doc_lemmas,
)

In [ ]:
query = "software bug"
mhs.print_search_results(query, topn=10)

With these results:

```console
http://dbpedia.org/resource/Leap_second?dbpv=2020-07&nif=sentence_23379_23433
'Cloudflare was affected by a leap second software bug.'
- Score 0.0000: 
- Full matches:
    'software'
    'bug'
http://dbpedia.org/resource/Leap_second?dbpv=2020-07&nif=sentence_23264_23378
'Leap second software bugs have affected the Altea airlines reservation system used by Qantas and Virgin Australia.'
- Score 0.0000: 
- Full matches:
    'software'
    'bugs'
http://dbpedia.org/resource/Leap_second?dbpv=2020-07&nif=sentence_20412_20600
'Older versions of Motorola Oncore VP, UT, GT, and M12 GPS receivers had a software bug that would cause a single timestamp to be off by a day if no leap second was scheduled for 256 weeks.'
- Score 0.0000: 
- Full matches:
    'software'
    'bug'
http://dbpedia.org/resource/Leap_second?dbpv=2020-07&nif=sentence_20104_20280
'Other reported software problems associated with the leap second\nA number of organizations reported problems caused by flawed software following the June 30, 2012, leap second.'
- Score 0.2759: 
- Full matches:
    'software'
- Close matches:
    'bug' -> 'problems' (0.8000)
    'bug' -> 'flawed' (0.8889)
    'bug' -> 'leap second' (0.9630)
    'bug' -> 'Other' (0.9655)
    'bug' -> 'software' (0.9655)
    'bug' -> 'organizations' (0.9655)
http://dbpedia.org/resource/JavaScript?dbpv=2020-07&nif=sentence_36105_36266
'In Windows Vista, Microsoft has attempted to contain the risks of bugs such as buffer overflows by running the Internet Explorer process with limited privileges.'
- Score 0.2759: 
- Full matches:
    'bugs'
- Close matches:
    'software' -> 'privileges' (0.8889)
    'software' -> 'Windows' (0.9286)
    'software' -> 'Internet' (0.9286)
    'software' -> 'Internet Explorer' (0.9286)
    'software' -> 'Windows Vista' (0.9474)
    'software' -> 'attempted' (0.9655)
    'software' -> 'bugs' (0.9655)
    'software' -> 'overflows' (0.9655)
    'software' -> 'process' (0.9655)
http://dbpedia.org/resource/Leap_second?dbpv=2020-07&nif=sentence_20766_21050
'Older Trimble GPS receivers had a software flaw that would insert a leap second immediately after the GPS constellation started broadcasting the next leap second insertion time (some months in advance of the actual leap second), rather than waiting for the next leap second to happen.'
- Score 0.2759: 
- Full matches:
    'software'
- Close matches:
    'bug' -> 'receivers' (0.8889)
    'bug' -> 'flaw' (0.8889)
    'bug' -> 'constellation' (0.8889)
    'bug' -> 'leap second' (0.9630)
    'bug' -> 'software' (0.9655)
    'bug' -> 'next' (0.9655)
    'bug' -> 'time' (0.9655)
    'bug' -> 'months' (0.9655)
http://dbpedia.org/resource/JavaScript?dbpv=2020-07&nif=sentence_30260_30359
'Most JavaScript-related security bugs are breaches of either the same origin policy or the sandbox.'
- Score 0.3103: 
- Full matches:
    'bugs'
- Close matches:
    'software' -> 'Java' (0.8889)
    'software' -> 'Script' (0.8889)
    'software' -> 'Java Script' (0.9286)
    'software' -> 'security' (0.9655)
    'software' -> 'bugs' (0.9655)
    'software' -> 'same' (0.9655)
    'software' -> 'origin' (0.9655)
    'software' -> 'policy' (0.9655)
http://dbpedia.org/resource/Leap_second?dbpv=2020-07&nif=sentence_22806_22962
'Despite the publicity given to the 2015 leap second, a small number of network failures occurred due to leap second-related software errors of some routers.'
- Score 0.3448: 
- Full matches:
    'software'
- Close matches:
    'bug' -> 'network' (0.8462)
    'bug' -> 'errors' (0.9286)
    'bug' -> 'routers' (0.9286)
    'bug' -> 'leap second' (0.9630)
    'bug' -> 'publicity' (0.9655)
    'bug' -> 'failures' (0.9655)
    'bug' -> 'software' (0.9655)
http://dbpedia.org/resource/Leap_second?dbpv=2020-07&nif=sentence_23120_23263
'Several older versions of the Cisco Systems NEXUS 5000 Series Operating System NX-OS (versions 5.0, 5.1, 5.2) are affected by leap second bugs.'
- Score 0.3448: 
- Full matches:
    'bugs'
- Close matches:
    'software' -> 'Operating System' (0.8000)
    'software' -> 'leap second' (0.9231)
    'software' -> 'Systems' (0.9286)
    'software' -> 'System' (0.9286)
    'software' -> 'older' (0.9655)
    'software' -> 'Series' (0.9655)
    'software' -> 'second' (0.9655)
    'software' -> 'bugs' (0.9655)
http://dbpedia.org/resource/JavaScript?dbpv=2020-07&nif=sentence_10024_10156
'Electron, Cordova, and other software frameworks have been used to create many applications with behavior implemented in JavaScript.'
- Score 0.3448: 
- Full matches:
    'software'
- Close matches:
    'bug' -> 'Electron' (0.8889)
    'bug' -> 'Script' (0.8889)
    'bug' -> 'other' (0.9655)
    'bug' -> 'software' (0.9655)
    'bug' -> 'applications' (0.9655)
```